In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

llm = OllamaLLM(
    model="qwen2.5-coder:7b",
    temperature=0
)

### Json Prompt Generation using JsonPromptParser w/ Pydantic

In [6]:
from pydantic import BaseModel, Field
from enum import Enum
from typing import List

class Sentiment(str, Enum):
    positive = "POSITIVE"
    negative = "NEGATIVE"
    neutral = "NEUTRAL"


class ProductReviewAnalytics(BaseModel):
    sentiment: Sentiment = Field(description="gives sentiment value for the review")
    themes: List[str] = Field(description="themes expressed from the review")

In [12]:
parser = JsonOutputParser(pydantic_object=ProductReviewAnalytics)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [13]:
prompt = PromptTemplate(
    template=""" Give sentiment and themes for a review.\n{formatted_instructions}\nInput Review:\n```\n{input_var}\n```""",
    input_variables=["input_var"],
    partial_variables={
        "formatted_instructions": parser.get_format_instructions()
    }
)

In [21]:
chain = prompt | llm | parser

In [22]:
response = chain.invoke(input={
    "input_var": "This bedsheet is beautiful and looks just like in the picture. The print is very pretty. It is really comfortable and smooth. Fabric is very nice , color doesn’t fade even after the wash. It fits our queen size bed perfectly ( it can be easy used for King size bed) . Go for it, value for money."
})

In [23]:
response

{'sentiment': 'POSITIVE',
 'themes': ['quality', 'comfort', 'value for money', 'fit', 'print']}